<a href="https://colab.research.google.com/github/greensky0107/self_study/blob/main/Day55_WordEmbedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectorization - One-Hot Encoding

In [88]:
!pip install konlpy

In [89]:
!pip install nltk

In [90]:
!pip install gensim

In [92]:
import re
from konlpy.tag import Okt
from collections import Counter
print("임포트 완료")

임포트 완료


In [93]:
text = "임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어."
text

'임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어.'

데이터 전처리

정규 표현식 (regex : [^ㄱ-ㅎㅏ-ㅣ가-힣 ])을 사용해서, 특수문자 제거 (한글과 공백은 유지)

In [94]:
reg = re.compile("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]")
text = reg.sub('', text)
print(text)

임금님 귀는 당나귀 귀 임금님 귀는 당나귀 귀 실컷 소리치고 나니 속이 확 뚫려 살 것 같았어


형태소 분석기인 Okt를 사용해서 tokenization

In [95]:
okt=Okt()
tokens = okt.morphs(text)
print(tokens)

['임금님', '귀', '는', '당나귀', '귀', '임금님', '귀', '는', '당나귀', '귀', '실컷', '소리', '치고', '나니', '속이', '확', '뚫려', '살', '것', '같았어']


단어장(Vocabulary) 만들기

빈도수가 높은 단어일수록 낮은 정수를 부여하므로, 파이썬의 Counter subclass를 사용해서 각 단어의 빈도수를 카운트

In [96]:
vocab = Counter(tokens)
print(vocab)

Counter({'귀': 4, '임금님': 2, '는': 2, '당나귀': 2, '실컷': 1, '소리': 1, '치고': 1, '나니': 1, '속이': 1, '확': 1, '뚫려': 1, '살': 1, '것': 1, '같았어': 1})


In [97]:
vocab['임금님']

2

In [98]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[('귀', 4), ('임금님', 2), ('는', 2), ('당나귀', 2), ('실컷', 1)]


In [99]:
word2idx={word[0] : index+1 for index, word in enumerate(vocab)}
print(word2idx) # 최종 단어장 word2idx

{'귀': 1, '임금님': 2, '는': 3, '당나귀': 4, '실컷': 5}


One-Hot Encoding 함수를 만들어서 단어의 one-hot vector찾기 실습

In [100]:
def one_hot_encoding(word, word2index):
       one_hot_vector = [0]*(len(word2index))
       index = word2index[word]
       one_hot_vector[index-1] = 1
       return one_hot_vector
print("슝=3")

슝=3


In [101]:
one_hot_encoding("임금님", word2idx)

[0, 1, 0, 0, 0]

Keras를 통해서 One-Hot Encoding

In [102]:
from tensorflow.keras.preprocessing.text import Tokenizer # Tokenizer는 단어장 만드는 도구
from tensorflow.keras.utils import to_categorical # to_categorical은 One-Hot Encoding 도구
print("임포트 완료")

임포트 완료


In [103]:
text = [['강아지', '고양이', '강아지'],['애교', '고양이'], ['컴퓨터', '노트북']]
text

[['강아지', '고양이', '강아지'], ['애교', '고양이'], ['컴퓨터', '노트북']]

In [104]:
t = Tokenizer()
t.fit_on_texts(text) # 주어진 텍스트로부터 단어장을 만들고
print(t.word_index) # 각 단어에 대한 인코딩 결과 출력.

{'강아지': 1, '고양이': 2, '애교': 3, '컴퓨터': 4, '노트북': 5}


In [105]:
vocab_size = len(t.word_index) + 1
print(vocab_size)

# vocab_size를 구할 때 1을 더해주는 이유는
# 실제로 자연어 처리를 할 때는 0번 단어가 특별 토큰(패딩(padding) 작업을 위한 패딩 토큰)으로 단어장에 추가되는 경우가 많기 때문

6


In [106]:
sub_text = ['강아지', '고양이', '강아지', '컴퓨터']
encoded = t.texts_to_sequences([sub_text])
print(encoded)  # text sequence를 integer sequence로 바꿔줌.

[[1, 2, 1, 4]]


In [107]:
one_hot = to_categorical(encoded, num_classes = vocab_size)
print(one_hot) # One-Hot Vector sequence로 변환

[[[0. 1. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0.]]]


# Word Embedding - Word2Vec (English)

In [109]:
import nltk
nltk.download('abc')

## 이부분 punkt 대신 punkt_tab을 이용
nltk.download('punkt_tab')

[nltk_data] Downloading package abc to /root/nltk_data...
[nltk_data]   Package abc is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [110]:
from nltk.corpus import abc
corpus = abc.sents()
print("슝~")

슝~


In [111]:
print(corpus[:3])

[['PM', 'denies', 'knowledge', 'of', 'AWB', 'kickbacks', 'The', 'Prime', 'Minister', 'has', 'denied', 'he', 'knew', 'AWB', 'was', 'paying', 'kickbacks', 'to', 'Iraq', 'despite', 'writing', 'to', 'the', 'wheat', 'exporter', 'asking', 'to', 'be', 'kept', 'fully', 'informed', 'on', 'Iraq', 'wheat', 'sales', '.'], ['Letters', 'from', 'John', 'Howard', 'and', 'Deputy', 'Prime', 'Minister', 'Mark', 'Vaile', 'to', 'AWB', 'have', 'been', 'released', 'by', 'the', 'Cole', 'inquiry', 'into', 'the', 'oil', 'for', 'food', 'program', '.'], ['In', 'one', 'of', 'the', 'letters', 'Mr', 'Howard', 'asks', 'AWB', 'managing', 'director', 'Andrew', 'Lindberg', 'to', 'remain', 'in', 'close', 'contact', 'with', 'the', 'Government', 'on', 'Iraq', 'wheat', 'sales', '.']]


In [112]:
print('코퍼스의 크기 :',len(corpus))

코퍼스의 크기 : 29059


In [113]:
from gensim.models import Word2Vec

model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=1, workers=4, sg=0)
print("모델 학습 완료!")

# vector size = 학습 후 임베딩 벡터의 차원
# window = 컨텍스트 윈도우 크기
# min_count = 단어 최소 빈도수 제한 (빈도가 적은 단어들은 학습 안함)
# workers = 학습을 위한 프로세스 수
# sg = 0은 CBoW, 1은 Skip-gram.

모델 학습 완료!


In [114]:
model_result = model.wv.most_similar("man")
print(model_result)
# man과 유사한 단어들은 skull, third, diet, Jupiter, woman, rally, star, fossil, population, tsunamis)

[('skull', 0.9485547542572021), ('third', 0.9377119541168213), ('diet', 0.9321730136871338), ('Jupiter', 0.9319924712181091), ('woman', 0.9310079216957092), ('rally', 0.930056095123291), ('star', 0.9294163584709167), ('fossil', 0.9287089705467224), ('population', 0.9255211353302002), ('tsunamis', 0.9237897992134094)]


In [115]:
# 모델을 저장하고, 이후 불러오는 방법
from gensim.models import KeyedVectors

### 코랩이라 경로 바꿔줘야합니다
model.wv.save_word2vec_format('./w2v')
loaded_model = KeyedVectors.load_word2vec_format("./w2v")
print("모델  load 완료!")

모델  load 완료!


In [116]:
model_result = loaded_model.most_similar("man")
print(model_result)

[('skull', 0.9485547542572021), ('third', 0.9377119541168213), ('diet', 0.9321730136871338), ('Jupiter', 0.9319924712181091), ('woman', 0.9310079216957092), ('rally', 0.930056095123291), ('star', 0.9294163584709167), ('fossil', 0.9287089705467224), ('population', 0.9255211353302002), ('tsunamis', 0.9237897992134094)]


# Word2Vec의 OOV(Out of Vocabulary) 문제

In [117]:
# 에러나는 코드들 (에러가 나더라도 놀라지 마세요)
# loaded_model.most_similar('overacting')
# loaded_model.most_similar('memorry')

In [118]:
# word2vec 모델 메타정보 및 텐서 내보내기
!python -m gensim.scripts.word2vec2tensor --input ./w2v --output ./w2v

2025-06-17 08:16:30,787 - word2vec2tensor - INFO - running /usr/local/lib/python3.11/dist-packages/gensim/scripts/word2vec2tensor.py --input ./w2v --output ./w2v
2025-06-17 08:16:30,787 - keyedvectors - INFO - loading projection weights from ./w2v
2025-06-17 08:16:32,704 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (31885, 100) matrix of type float32 from ./w2v', 'binary': False, 'encoding': 'utf8', 'datetime': '2025-06-17T08:16:32.702510', 'gensim': '4.3.3', 'python': '3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]', 'platform': 'Linux-6.1.123+-x86_64-with-glibc2.35', 'event': 'load_word2vec_format'}
2025-06-17 08:16:34,773 - word2vec2tensor - INFO - 2D tensor file saved to ./w2v_tensor.tsv
2025-06-17 08:16:34,773 - word2vec2tensor - INFO - Tensor metadata file saved to ./w2v_metadata.tsv
2025-06-17 08:16:34,776 - word2vec2tensor - INFO - finished running word2vec2tensor.py


# Fast Text

In [122]:
from gensim.models import FastText
fasttext_model = FastText(corpus, window=5, min_count=5, workers=4, sg=1)
print("FastText 학습 완료!")

FastText 학습 완료!


In [123]:
fasttext_model.wv.most_similar('overacting')

[('extracting', 0.9459770321846008),
 ('overwhelming', 0.9357720017433167),
 ('lifting', 0.9355456829071045),
 ('attracting', 0.9335088133811951),
 ('resolving', 0.9326339960098267),
 ('stealing', 0.9313978552818298),
 ('emptying', 0.9310289025306702),
 ('tasting', 0.9284043312072754),
 ('fluctuating', 0.9276565909385681),
 ('negotiating', 0.92735755443573)]

In [124]:
fasttext_model.wv.most_similar('memoryy')

[('memory', 0.9523078203201294),
 ('intelligence', 0.8751842975616455),
 ('musical', 0.8634611368179321),
 ('interactive', 0.8557308912277222),
 ('music', 0.855712354183197),
 ('basic', 0.8532955050468445),
 ('intermediate', 0.8525428771972656),
 ('intercourse', 0.8491395115852356),
 ('emotion', 0.8450659513473511),
 ('mechanism', 0.8442756533622742)]

# GloVe

In [125]:
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-50")  # glove vectors 다운로드
glove_model.most_similar("dog")  # 'dog'과 비슷한 단어 찾기

[('cat', 0.9218004941940308),
 ('dogs', 0.8513158559799194),
 ('horse', 0.7907583713531494),
 ('puppy', 0.7754920721054077),
 ('pet', 0.7724708318710327),
 ('rabbit', 0.7720814347267151),
 ('pig', 0.7490062117576599),
 ('snake', 0.7399188876152039),
 ('baby', 0.7395570278167725),
 ('bite', 0.7387937307357788)]

In [129]:
glove_model.most_similar('overacting')

[('impudence', 0.7842012047767639),
 ('puerile', 0.7816032767295837),
 ('winningly', 0.7644237875938416),
 ('grossness', 0.7576098442077637),
 ('deconstructions', 0.748936653137207),
 ('over-the-top', 0.7460805773735046),
 ('buffoonery', 0.746045708656311),
 ('impetuosity', 0.7415392398834229),
 ('sophomoric', 0.736961841583252),
 ('zaniness', 0.7353197336196899)]

In [130]:
# glove_model.most_similar('memoryy')
# memoryy는 인식하지 못함